# Creating Loan Offers with Feature Engineering
## Practice Skeleton

**Short name (GitHub):** `LoanOrig`

**Adaptation of:** `LaptopPrice` / `TaxCreate` (unit-bearing strings → numeric features → encode → Random Forest).

**Card:** `data/loan_applications.csv` — **920** draft applications × **24** columns. Target `offer_amount` is the principal the desk *creates* (USD; mean ≈ $53,242, median $20,024, long mortgage tail). Income, debt, request, down payment, reserves, note rate, term, borrowers, stars, and vintage arrive as text (`"$72,000"`, `"7.25%"`, `"36-month"`, `"2 borrowers"`, `"4 stars"`, `"2023 VY"`).

**How to use**
- Fill cells marked `# YOUR CODE HERE`. Keep the cheat-sheet and flowchart visible.
- Compare with `LoanOrig_Solution.ipynb` only after an attempt.
- Data: `data/loan_applications.csv`. Charts: `loanorig_*.png`.
- Clone with `LoanOrig_Reusable_Template.ipynb`.
- **Not a credit decision, not lending advice, not an origination system.** Teaching book only.


## Inline cheat-sheet (keep this cell visible)

See also **`LoanOrig_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Money strings | `.astype(str).str.replace('$','').str.replace(',','').astype(float)` |
| Percent | strip `'%'` then `/ 100` |
| Composite | `net_capacity = annual_income_usd - other_debt_usd` |
| Borrowers / term | extract digits; strip `'-month'` |
| Sentinel vintage | `'Not Available'` → `'0'` *before* stripping `' VY'` |
| Cardinality | `< 5` → one-hot `drop_first`. `≥ 5` → target-mean `offer_amount` |
| Leakage | Fit means on **train only** |
| Split / model | `test_size=0.2`, `random_state=42` / `RandomForestRegressor(random_state=42)` |
| Metrics | MAE in dollars + R² vs the **mean-offer baseline** |
| Watch | `requested_usd` is the applicant's ask. Policy ≈ min(request, capacity), so it will dominate impurity. Drop it in practice cell 2. |
| Never | Treat this as an approve/reject engine (see the Banking LogReg packs for that). |


## Flowchart of the desired outcome

![LoanOrig flow](loanorig_flowchart.png)

Load the 920-row book → strip `$` `,` `%` → engineer `net_capacity` → clean stars, borrowers, term, vintage → EDA → encode by cardinality → 80/20 → forest vs linear / mean → MAE + R² + top features → simulation knobs.


## 0. Packages


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
sns.set_theme(style="whitegrid")


## 1. Load and inspect

Print `.head()`, `.info()`, `.shape`, `.nunique()`, `df['offer_amount'].describe()`.


In [ ]:
# YOUR CODE HERE
df = None


## 2. Parse money columns and engineer net capacity

Loop over `['annual_income_usd','other_debt_usd','requested_usd','down_payment_usd','reserves_usd']`.
Strip `$` and `,`. Create `net_capacity = annual_income_usd - other_debt_usd`.


In [ ]:
# YOUR CODE HERE


## 3. Clean rate, stars, vintage, borrowers, term

- `note_rate`: strip `%` → float / 100.
- `bureau_stars`: strip `" stars"` then `" star"` → int.
- `orig_vintage`: `"Not Available"` → `"0"`, strip `" VY"` → `orig_year_n`, drop original.
- `borrowers`: extract digits → int.
- `term_months`: strip `"-month"` → int.


In [ ]:
# YOUR CODE HERE


## 4. EDA

Heatmap; boxplot borrowers vs offer; boxplot bureau_stars vs offer; histogram of offer (mortgage tail).

![heatmap](loanorig_heatmap.png)
![borrowers](loanorig_borrowers_box.png)
![stars](loanorig_stars_box.png)


In [ ]:
# YOUR CODE HERE
numeric_df = None


## 5. Encode + split

Remaining objects: `branch`, `product_class`, `employment_status` (high-card) and channel / purpose / complexity / timing / digital / existing / resident / collateral (low-card).

Lesson version may encode on the full frame — note the leakage. Stretch: split first.


In [ ]:
# YOUR CODE HERE
X_train = X_test = y_train = y_test = None


## 6. Fit the Random Forest


In [ ]:
# YOUR CODE HERE
rf_model = None
y_pred = None


## 7. Evaluate and rank features

Print MAE, RMSE, R², mean-offer baseline.

![importance](loanorig_importance.png)
![pred vs actual](loanorig_pred_actual.png)


In [ ]:
# YOUR CODE HERE
mae = r2 = None


## 8. Alternate code


### 8a. Regex extract for money


In [ ]:
# YOUR CODE HERE


### 8b. Leakage-safe target encoding


In [ ]:
# YOUR CODE HERE


### 8c. Linear / Ridge


In [ ]:
# YOUR CODE HERE


### 8d. Permutation importance


In [ ]:
# YOUR CODE HERE


## 9. More practice

1. `log1p(offer_amount)` target, `expm1` predictions.
2. **Drop `requested_usd`** (the ask). How much R² remains from capacity alone?
3. Fit only `product_class == 'Personal'`.
4. Tail MAE on offers ≥ $80,000 (mostly mortgage / SME).
5. Policy rebuild: `min(requested, 0.35 * income * term/12)` vs the forest.


In [ ]:
# YOUR CODE HERE — pick at least two


## 10. Simulation

![simulation](loanorig_simulation.png)

Reference: MAE ≈ **$5,153**, R² ≈ **0.982**.


In [ ]:
N_EST = 100
MAX_DEPTH = None
NOISE_SD = 0
SUBSAMPLE = 1.0
RANDOM_STATE = 42


In [ ]:
# YOUR CODE HERE


## 11. What this model can and cannot do

**Can**
- Reconstruct a noisy min(request, capacity) offer policy from parsed application fields.
- Beat a mean-offer guess by an order of magnitude (~$5.2k vs ~$54.7k MAE).
- Show that product, term, and rate still move residuals after the ask is known.

**Cannot**
- Approve or decline a borrower (wrong task — use the Banking LogReg packs).
- Set a compliant APR or meet fair-lending constraints.
- Use inquiry/trade counts as if they were known at first touch for a thin file.
- Price a 2026 product book without a refresh.

**Same pipeline:** deposit opening amounts, limit increases, insurance sum-assured, trade-finance facilities.
